In [ ]:
import torch
import torch.optim as optim
from dataset import get_pennfudan_dataloaders
from models import ObjectCountingModel
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from settings import DATA_DIR
from typing import Literal
import os
from evaluate import evaluate_model

We implemented the following changes with respect to the original training script:
- Create dataloaders outside training function to optimize resources.
- Add the model version to the `ObjectCountingModel` class and training and evaluation functions.
- Save the best and last model checkpoints for each version for a fair comparison between versions to `checkpoints`
- Add early stopping

In [ ]:
def train_model(
    train_loader,
    val_loader,
    category_names,
    num_epochs=10,
    learning_rate=0.005,
    version: Literal["v1", "v2"] = "v2",
    patience=5,
    min_delta=1e-4,
):
    device = "cuda" if torch.cuda.is_available() else "cpu"

    model = ObjectCountingModel(num_classes=2, version=version)
    model.to(device)

    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = optim.SGD(params, lr=learning_rate, momentum=0.3, weight_decay=0.0005)
    lr_scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

    map_metric = MeanAveragePrecision(iou_type="bbox").to(device)
    best_map = float("-inf")
    model_patience = patience

    os.makedirs("checkpoints", exist_ok=True)

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0

        for images, targets in train_loader:
            images = [image.to(device) for image in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())

            optimizer.zero_grad()
            losses.backward()
            optimizer.step()

            running_loss += losses.item()

        lr_scheduler.step()

        model.eval()
        map_metric.reset()  # Reset for each epoch
        with torch.no_grad():
            for images, targets in val_loader:
                images = [image.to(device) for image in images]
                targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

                preds = model(images)
                map_metric.update(preds, targets)

        metrics = map_metric.compute()
        current_map = float(metrics["map"].item())
        current_map50 = float(metrics["map_50"].item())

        epoch_str = f"Epoch {epoch + 1}/{num_epochs}"
        loss_str = f"Loss: {running_loss / len(train_loader):.4f}"
        map_str = f"Val mAP: {current_map:.4f} (mAP@0.5: {current_map50:.4f})"
        print(f"{epoch_str}, {loss_str}\n {map_str}")

        # Early stopping and best checkpoint update.
        if current_map > best_map + min_delta:
            best_map = current_map
            model_patience = patience
            torch.save(
                model.state_dict(),
                f"checkpoints/object_counting_model_{version}_best.pth",
            )
        else:
            model_patience -= 1

        # Always save current epoch state as latest/last checkpoint.
        torch.save(
            model.state_dict(),
            f"checkpoints/object_counting_model_{version}.pth",
        )

        if model_patience == 0:
            print(f"Early stopping triggered. Best mAP: {best_map:.4f}")
            break

    return model, category_names

In [ ]:
batch_size = 8
train_loader, val_loader, category_names = get_pennfudan_dataloaders(
    batch_size, root=str(DATA_DIR / "PennFudanPed")
)

In [ ]:
shared_params = {
    "num_epochs": 100,
    "train_loader": train_loader,
    "val_loader": val_loader,
    "category_names": category_names,
    "learning_rate": 0.005,
    "patience": 10
}

In [ ]:
model_v1 = train_model(**shared_params, version="v1")

In [ ]:
evaluate_model(model_path="checkpoints/object_counting_model_v1_best.pth", version="v1")

In [ ]:
model_v2 = train_model(**shared_params, version="v2")

In [ ]:
evaluate_model(model_path="checkpoints/object_counting_model_v2_best.pth", version="v2")

- Tiempo de entrenamiento: 
- Tiempo de ejecución:
- Best mAP@0.5: 